**BBM473 Database Laboratory (Spring 2025)**


Exercise 4: Design Theory
------------



## Activity 4.1: FDs & Closures
------

Recall that given a set of attributes  $\{A_1, \dots, A_n\}$ and a set of FDs $\Gamma$

The closure, denoted $\{A_1, \dots, A_n\}^+$, is defined to be the largest set of attributes B s.t. $$A_1,\dots,A_n \rightarrow B \text{ using } \Gamma.$$

We've built some functions to compute the closure of a set of attributes and other such operations (_feel free to look at the code- it's pretty simple and clean, if we say so ourselves..._):

In [1]:
from closure import compute_closure

### Task 1

Consider a schema with attributes $X=\{A,B,C,D,E,F,G,H\}$.

In this exercise, you are given a set of attributes $A\subset X$ and a set of FDs $F$.  Find **one FD** to add to $F$ so that the closure $A^+=X$

**Note: you can add FDs to the below set $F$ using e.g. `F.append((set([...]), set([...])))` and then check how you're doing using the `compute_closure` function from above!**

(As we'll find out immediately after this activity, this equivalent to saying: _Find one FD to add such that $A$ becomes a superkey for $X$_)

In [2]:
A = set(['A', 'B','F'])
F = [(set(['A', 'C']), 'D'),
     (set(['D','H', 'G']), 'E'),
     (set(['A', 'B']), 'G'),
     (set(['F', 'B', 'G']), 'C')]

In [3]:
compute_closure(A,F)

{'A', 'B', 'C', 'D', 'F', 'G'}

Write the FD below:

In [4]:
F.append((set(['A', 'B', 'F']), set(['H']))) # if you enter an element with multiple attributes use set()
compute_closure(A,F)

{'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'}

## Activitity 4.2: Superkeys & Keys (30 pts.)

Next, we'll add some new functions now for finding superkeys and keys.  Recall:
* A _superkey_ for a relation $R(B_1,\dots,B_m)$ is a set of attributes $\{A_1,\dots,A_n\}$ s.t.
$$ \{A_1,\dots,A_n\} \rightarrow B_{j} \text{ for all } j=1,\dots m$$
* A _key_ is a minimal (setwise) _superkey_

The algorithm to determine if a set of attributes $A$ is a superkey for $X$ is actually very simple- we just see if the $A^+=X$:

In [5]:
def is_superkey_for(A, X, fds, verbose=False):
    return X.issubset(compute_closure(A, fds, verbose=verbose))

Then, to check if $A$ is a key for $X$, we just confirm that:
* (a) it is a superkey
* (b) there are no smaller superkeys (_Note that we only need to check for superkeys of one size smaller- think about why!_)

In [6]:
import itertools
def is_key_for(A, X, fds, verbose=False):
    subsets = set(itertools.combinations(A, len(A)-1))
    return is_superkey_for(A, X, fds) and \
        all([not is_superkey_for(set(SA), X, fds) for SA in subsets])

### Task 1 (20 pts.)

Given the schema $R=\{A,B,C\}$, define a set of FDs such that there are two- _and only two_- keys, and check using the above functions!

In [7]:
R = set(['A','B','C'])

Write your answer below:

In [8]:
# fd1 = (___, ___) # ___ -> ___
# fd2 = (___, ___) # ___ -> ___

fd1 = (set(['B']), set(['C']))  # B -> C
fd2 = (set(['C']), set(['B']))  # C -> B

F = [fd1, fd2]

# The first sets of fd1 and fd2 tuples should be keys
print(is_key_for(set(['A', 'B']), R, F))
print(is_key_for(set(['A', 'C']), R, F))

True
True


In [9]:
print(is_key_for(set(['B', 'C']), R, F))
print(is_key_for(set(['A']), R, F))

False
False


### Task 2 (10 pts.)

Now, given the below relation $R$, and the above tools, define a set of FDs to result in the most keys possible.  How many keys can you make?  Largest number wins it all!

_Bonus question: how many different sets of FDs will result in this maximum number of keys?_

In [10]:
R = set(['A','B','C','D','E'])

In [11]:
F = [
    (set(['A', 'B']), set(['C', 'D', 'E'])),
    (set(['A', 'C']), set(['B', 'D', 'E'])),
    (set(['A', 'D']), set(['B', 'C', 'E'])),
    (set(['A', 'E']), set(['B', 'C', 'D'])),
    (set(['B', 'C']), set(['A', 'D', 'E'])),
    (set(['B', 'D']), set(['A', 'C', 'E'])),
    (set(['B', 'E']), set(['A', 'C', 'E'])),
    (set(['C', 'D']), set(['A', 'B', 'E'])),
    (set(['C', 'E']), set(['A', 'B', 'D'])),
    (set(['D', 'E']), set(['A', 'B', 'C'])),
]

any subset of size 2 from R is a key, total 10 keys

## Activity 4.3: BCNF Decompositions (30 pts.)

The goal for this activity will be to compute some BCNF decompositions, using the tools from ```closure.py```

First we'll load those tools, and some sample data:

In [12]:
from closure import compute_closure, display_side_by_side, print_setup

In [13]:
%load_ext sql
%sql sqlite://
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [14]:
%%sql DROP TABLE IF EXISTS T;
CREATE TABLE T(course VARCHAR, classroom INT, time INT);
INSERT INTO T VALUES ('BBM471', 132, 900);
INSERT INTO T VALUES ('BBM473', 140, 1000);
INSERT INTO T VALUES ('AIT204', 210, 900);

 * sqlite://
Done.
Done.
1 rows affected.
1 rows affected.
1 rows affected.


[]

In [15]:
%sql SELECT * FROM T;

 * sqlite://
Done.


course,classroom,time
BBM471,132,900
BBM473,140,1000
AIT204,210,900


### Task 1 (10 pts.)

First, let's decompose `T` into BCNF!  Explicitly go through the steps of the BCNF algorithm using the `compute_closure` function, then decompose the following table (i.e. by creating new SQL tables) into BCNF:

We've also made a function, `display_side_by_side`, for nicer display!

In [16]:
%sql SELECT * FROM T;

 * sqlite://
Done.


course,classroom,time
BBM471,132,900
BBM473,140,1000
AIT204,210,900


We are given the following FDs:

In [17]:
A = set(['course', 'classroom', 'time'])
F = [('course', 'classroom'), (set(['classroom', 'time']), 'course')]
print_setup(A, F)

Attributes = {classroom,course,time}
FDs:

=> 	course -> classroom

=> 	classroom,time -> course


**Q:** What real-world constraints do these FDs express? Answer below: (In one or two sentences)

Each course is mapped to exactly one classroom.

The combination of a classroom and time together uniquely determines the course

Now, use the `compute_closure` function to help decompose this table to BCNF:

In [18]:
compute_closure(set(["course"]), F)

{'classroom', 'course'}

Not all attributes -> Not a superkey

In [19]:
compute_closure(set(["classroom", "time"]), F) # {'classroom', 'course', 'time'} -> superkey

{'classroom', 'course', 'time'}

Compose into two tables, $T_1$ and $T_2$:

In [20]:
%%sql
DROP TABLE IF EXISTS T1;
CREATE TABLE T1 AS SELECT DISTINCT * FROM (
        SELECT course, classroom FROM T
);

 * sqlite://
Done.
Done.


[]

In [21]:
%%sql
DROP TABLE IF EXISTS T2;
CREATE TABLE T2 AS SELECT DISTINCT * FROM (
        SELECT course, time FROM T
);

 * sqlite://
Done.
Done.


[]

Now run the below to display the decomposed tables side-by-side:

In [22]:
l = %sql SELECT * FROM T1;
r = %sql SELECT * FROM T2;
display_side_by_side(l,r)

 * sqlite://
Done.
 * sqlite://
Done.


course classroom BBM471 132 BBM473 140 AIT204 210 course time BBM471 900 BBM473 1000 AIT204 900

**Q:** Is this now in BCNF?

course is a superkey in T1 => T1 is in BCNF

No non-trivial FDs in T2 (classroom is not in T2) and course -> time is not given nor implied so T2 is automatically in BCNF

decomposition is now in BCNF

### Task 2 (20 pts.)

In the next section of lecture, we'll discuss a shortcoming of BCNF decompositions; let's see if we can get a glimpse of this now.

See if you can insert rows into $T_1$ and/or $T_2$ _which respect the local FDs that still hold_, such that **when $T_1$ and $T_2$ are now recomposed, the original FDs do not hold!**

In [23]:
%%sql
INSERT INTO T1 VALUES ('BBM342', 132);

INSERT INTO T2 VALUES ('BBM342', 900);


 * sqlite://
1 rows affected.
1 rows affected.


[]

Now, reconstruct and print the re-composed table using a SQL query:

In [24]:
%%sql
SELECT * FROM T1 JOIN T2 USING (course);

 * sqlite://
Done.


course,classroom,time
BBM471,132,900
BBM473,140,1000
AIT204,210,900
BBM342,132,900


**Q:** What went wrong??  And how could we prevent this from occuring?

There are two rows with the same (classroom, time) but different course

violating the original FD: (classroom, time) -> course

Instead of decomposing to BCNF, decompose to 3NF to preserve all original FDs (dependency preservation):

T1(course, classroom)

T2(classroom, time, course) (classroom, time → course (includes candidate key))

## Activity 4.4 : MVDs (40 pts.)


First, execute the following codes:

In [25]:
# Utility functions...
from IPython.core.display import display_html, HTML
def to_html_table(res, style=None):
    html = '<table' + (' style="' + style + '"' if style else '') + '><tr><th>'
    html += '</th><th>'.join(res.keys) + '</th></tr><tr><td>'
    html += '</td></tr><tr><td>'.join(['</td><td>'.join([str(cell) for cell in row]) for row in list(res)])
    return html + '</tr></table>'
def display_side_by_side(l, r):
    s = "display: inline-block;"
    html = to_html_table(l, style=s) + ' ' + to_html_table(r, style=s)
    display_html(HTML(data=html))

### Task 1 (10 pts.)

**Formal definition**

Given a relation $R$ having attributes $A$, and two sets of attributes $X,Y\subseteq A$, the **_multi-value dependency (MVD)_** $X\twoheadrightarrow Y$ holds on $R$ if, for **any** tuples $t_1,t_2\in R$ s.t. $t_1[X] = t_2[X]$, there exists a tuple $t_3\in R$ s.t.:

* $t_3[X] = t_1[X] = t_2[X]$
* $t_3[Y] = t_1[Y]$
* $t_3[A\setminus Y] = t_2[A\setminus Y]$

where $A\setminus B$ = all elements of $A$ not in $B$.

So let's consider a toy example at this point:

In [26]:
%%sql
DROP TABLE IF EXISTS R; CREATE TABLE R (A INT, B INT, C INT);
INSERT INTO R VALUES (1, 1, 0);
INSERT INTO R VALUES (1, 0, 1);
SELECT * FROM R;

 * sqlite://
Done.
Done.
1 rows affected.
1 rows affected.
Done.


A,B,C
1,1,0
1,0,1


Let's consider the first two rows as $t_1$ and $t_2$ respectively; what is the $t_3$ that the definition tells us we must add if we want the MVD $\{A\}\twoheadrightarrow\{B\}$ to hold?  Let's add it in:

In [27]:
%sql INSERT INTO R VALUES (1,1,1); SELECT * FROM R;

 * sqlite://
1 rows affected.
Done.


A,B,C
1,1,0
1,0,1
1,1,1


However what about if we consider the first two rows as $t_2$ and $t_1$ respectively?  What row would the definition tell us we'd have to add in?

In [28]:
%sql INSERT INTO R VALUES (1,0,0); SELECT * FROM R;

 * sqlite://
1 rows affected.
Done.


A,B,C
1,1,0
1,0,1
1,1,1
1,0,0


### Task 2 (30 pts.)

We developed some appealingly simple python tools for this lecture, but let's switch back to SQL quickly- write a query which returns **no values _only if_** the MVD course to staff holds

$\{$course$\}\twoheadrightarrow\{$staff$\}$

Then, comment out the insert statement(s) above so that the MVD does hold (do you remember how to comment out lines in SQL from earlier activities?  It's super useful!)

First, execute the following codes:

In [29]:
%%sql
DROP TABLE IF EXISTS courses;
CREATE TABLE courses (course TEXT, staff TEXT, student TEXT);
INSERT INTO courses VALUES ('BBM473','Amy','Bob');
INSERT INTO courses VALUES ('BBM471','Chris','Deb');
INSERT INTO courses VALUES ('BBM471','Chris','Eli');
INSERT INTO courses VALUES ('BBM471','Firas','Deb');
INSERT INTO courses VALUES ('BBM471','Firas','Bob');
INSERT INTO courses VALUES ('BBM471','Firas','Eli');
-- INSERT INTO courses VALUES ('BBM471','Chris','Bob'); --  This is the missing one


 * sqlite://
Done.
Done.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
0 rows affected.


[]

write a query which returns **no values _only if_** the MVD course to staff holds

$\{$course$\}\twoheadrightarrow\{$staff$\}$

Now write your answer below:

In [30]:
%%sql
SELECT c1.course, c1.staff, c2.student
FROM courses c1, courses c2
WHERE c1.course = c2.course
  AND NOT EXISTS (
    SELECT 1 FROM courses c3
    WHERE c3.course = c1.course
      AND c3.staff = c1.staff
      AND c3.student = c2.student
  );


 * sqlite://
Done.


course,staff,student
BBM471,Chris,Bob
BBM471,Chris,Bob


Save and submit your notebook file (.ipynb) to the form below with your name and student ID. Do not include the database file or any other file. Only one notebook file is sufficient.

https://forms.gle/La5o5KJiWxbN9FHf9

**Note:** *You cannot submit your exercise notebook remotely. You have to be at the class.*